In [1]:
import requests
from bs4 import BeautifulSoup
from langdetect import detect
from urllib.parse import urlparse

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# Your URLs
urls = [
    "https://www.enabbaladi.net/785960/%D8%A8%D9%8A%D8%AA-%D8%AC%D9%86-%D8%A5%D8%B3%D8%B1%D8%A7%D8%A6%D9%8A%D9%84-%D8%AA%D9%87%D8%B1%D8%A8-%D8%A5%D9%84%D9%89-%D8%A7%D9%84%D8%A3%D9%85%D8%A7%D9%85-%D9%81%D9%8A-%D8%B3%D9%88%D8%B1%D9%8A/",
    "https://www.aljazeera.com/news/2025/11/30/israel-attacks-on-syria-what-happened-who-did-israel-claim-it-was-after",
    "https://www.lefigaro.fr/international/syrie-une-operation-israelienne-dans-le-sud-du-pays-fait-dix-morts-20251128",
    "https://www.derstandard.at/story/3000000298372/syrische-staatsmedien-zehn-tote-durch-israelische-angriffe-in-syrien",
    "https://www.larazon.es/internacional/menos-diez-muertos-ataques-israelies-afueras-damasco-celula-islamista_20251128692951786e5e5012dcd1db0e.html"
]


def scrape_article(url):
    """Scrape title + text content + metadata from a news article."""
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        # Source domain
        source = urlparse(url).netloc

        # Title extraction
        if soup.title:
            title = soup.title.get_text().strip()
        else:
            title = "No title found"

        # Extract readable text (all paragraph tags)
        paragraphs = soup.find_all("p")
        content = "\n".join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])

        # If content empty, fallback to all text
        if not content:
            content = soup.get_text(separator="\n")

        # Detect language
        try:
            language = detect(content[:500])   # detect using the first 500 chars
        except:
            language = "unknown"

        return {
            "source": source,
            "language": language,
            "title": title,
            "content": content
        }

    except Exception as e:
        return {
            "source": url,
            "error": str(e)
        }


# Master dictionary for all articles
articles_data = {}

for url in urls:
    articles_data[url] = scrape_article(url)

# Pretty print the result
import json
print(json.dumps(articles_data, ensure_ascii=False, indent=4))

{
    "https://www.enabbaladi.net/785960/%D8%A8%D9%8A%D8%AA-%D8%AC%D9%86-%D8%A5%D8%B3%D8%B1%D8%A7%D8%A6%D9%8A%D9%84-%D8%AA%D9%87%D8%B1%D8%A8-%D8%A5%D9%84%D9%89-%D8%A7%D9%84%D8%A3%D9%85%D8%A7%D9%85-%D9%81%D9%8A-%D8%B3%D9%88%D8%B1%D9%8A/": {
        "source": "www.enabbaladi.net",
        "language": "ar",
        "title": "بيت جن.. إسرائيل تهرب إلى الأمام في سوريا - عنب بلدي",
        "content": "آثار الغارات الإسرائيلية على بلدة بيت جن بريف دمشق - 28 تشرين الثاني 2025 (سانا)\nعنب بلدي – عمر علاء الدين\nأشعلت الاشتباكات بين شبان من بلدة بيت جن بريف دمشق والقوات الإسرائيلية، فجر الجمعة 28 من تشرين الثاني، مسار الأحداث في الجنوب السوري، إذ كانت إسرائيل تستمر بتوغلاتها دون ردود فعلية سوى اشتباكات سابقة مع أبناء بلدتي نوى وكويا بريف درعا.\nوقابلت القوات الإسرائيلية المقاومة المحلية بقصف المدفعية الثقيلة، ما أسفر عن 13 قتيلًا و24 مصابًا، بينما أصيب ستة ضباط إسرائيليين بجروح بالغة ومتوسطة وخفيفة، إلى جانب دمار عدة منازل وحركة نزوح لأهالي بيت جن إلى المناطق المجاورة.\nويأتي توقيت هذا التصعيد ع

## AWS Translate

In [2]:
# Import required libraries and set up our environment
from pprint import PrettyPrinter

import boto3

print("📚 Setting up the environment...")
pp = PrettyPrinter(indent=2)
translate = boto3.client("translate")
print("✅ Environment setup complete!")
print(f"🌍 Using AWS region: {translate.meta.region_name}")

📚 Setting up the environment...
✅ Environment setup complete!
🌍 Using AWS region: eu-west-1


In [3]:
def translate_long_text(text, source_lang, max_chunk_size=9000):
    """
    Translate long text by splitting it into chunks below AWS Translate's 10KB limit.
    """
    if not text or source_lang == "en":
        return text

    chunks = []
    current_chunk = ""

    # Split by sentences for cleaner translation
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sentence in sentences:
        if len((current_chunk + " " + sentence).encode("utf-8")) < max_chunk_size:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    translated_chunks = []

    for chunk in chunks:
        response = translate.translate_text(
            Text=chunk,
            SourceLanguageCode=source_lang,
            TargetLanguageCode="en"
        )
        translated_chunks.append(response["TranslatedText"])

    return "\n".join(translated_chunks)


In [5]:
translated_articles = {}

for url, article in articles_data.items():
    lang = article.get("language", "unknown")

    title = article.get("title", "")
    content = article.get("content", "")

    if lang != "en" and lang != "unknown":
        print(f"🔄 Translating article from {lang} → en: {url}")

        title_en = translate_long_text(title, lang)
        content_en = translate_long_text(content, lang)

    else:
        print(f"✔ Article already in English: {url}")
        title_en = title
        content_en = content

    translated_articles[url] = {
        "source": article.get("source"),
        "original_language": lang,
        "title_en": title_en,
        "content_en": content_en,
        "title_original": title,
        "content_original": content
    }

print("\n🎉 Translation complete!")


🔄 Translating article from ar → en: https://www.enabbaladi.net/785960/%D8%A8%D9%8A%D8%AA-%D8%AC%D9%86-%D8%A5%D8%B3%D8%B1%D8%A7%D8%A6%D9%8A%D9%84-%D8%AA%D9%87%D8%B1%D8%A8-%D8%A5%D9%84%D9%89-%D8%A7%D9%84%D8%A3%D9%85%D8%A7%D9%85-%D9%81%D9%8A-%D8%B3%D9%88%D8%B1%D9%8A/
✔ Article already in English: https://www.aljazeera.com/news/2025/11/30/israel-attacks-on-syria-what-happened-who-did-israel-claim-it-was-after
🔄 Translating article from fr → en: https://www.lefigaro.fr/international/syrie-une-operation-israelienne-dans-le-sud-du-pays-fait-dix-morts-20251128
🔄 Translating article from de → en: https://www.derstandard.at/story/3000000298372/syrische-staatsmedien-zehn-tote-durch-israelische-angriffe-in-syrien
🔄 Translating article from es → en: https://www.larazon.es/internacional/menos-diez-muertos-ataques-israelies-afueras-damasco-celula-islamista_20251128692951786e5e5012dcd1db0e.html

🎉 Translation complete!


In [8]:
print("\n📝 Translated Articles Summary:"  )
for url, article in translated_articles.items():
    print(f"🌐 Source: {article['source']}")
    print(f"🗣 Original Language: {article['original_language']}")
    print(f"📰 Title (EN): {article['title_en']}...")
    print(f"📰 content: {article['content_en']}...")
## AWS Translate


📝 Translated Articles Summary:
🌐 Source: www.enabbaladi.net
🗣 Original Language: ar
📰 Title (EN): Beit Jinn.. Israel flees forward in Syria - Enab Baladi...
📰 content: Effects of Israeli raids on the town of Beit Jinn in Damascus countryside - 28 November 2025 (SANA)
Enab Baladi — Omar Aladdin
At dawn on Friday, November 28, clashes between young men from the town of Beit Jinn in Damascus countryside and Israeli forces sparked the course of events in southern Syria. Israel was continuing its incursions without any reactions except previous clashes with the people of the towns of Nawa and Koya in Daraa countryside. Israeli forces met the local resistance with heavy artillery shelling, resulting in 13 deaths and 24 injuries, while six Israeli officers were seriously, moderately and lightly injured, in addition to the destruction of several houses and the displacement of Beit Jinn residents to nearby areas. The timing of this escalation follows a series of escalating measures taken by Is

## Comprehend

In [ ]:
pp = pprint.PrettyPrinter(indent=2)

# Create Comprehend client
comprehend = boto3.client(service_name="comprehend", region_name="eu-west-1")